In [1]:
# Ensure latest library code is loaded (helpful after edits)
import os
import sys
import time, math
from itertools import product

from libs.multilevel_squeme import (
    multilevel_bipartition,
    edge_cut,
    k_way_partition,
    edge_cut_kway,
    kway_balance_info,
    partition_graph_metis,
)

from libs.utils import (
    load_mtx,
    matrix_to_graph,
    summarize_generic
)

In [2]:
# Diagnostics: interpreter and METIS backend
print("Python executable:", sys.executable)

try:
    import pymetis
    print("PyMetis:", getattr(pymetis, "__file__", "?"))
except Exception as e:
    print("PyMetis import failed:", repr(e))

Python executable: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/.venv/bin/python
PyMetis: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/.venv/lib/python3.10/site-packages/pymetis/__init__.py


In [3]:
data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

k_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_k.mtx")
m_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_m.mtx")
print("Using data_dir:", data_dir)

Using data_dir: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/data


In [4]:
# Load matrices
A_K = load_mtx(k_matrix_path, 'K')
A_M = load_mtx(m_matrix_path, 'M')

if A_K is None or A_M is None:
    raise RuntimeError("Matrix load failed; check file paths printed above.")

print(f"Loaded: K | shape={A_K.shape}, nnz={A_K.nnz}")
print(f"Loaded: M | shape={A_M.shape}, nnz={A_M.nnz}")

Loaded: K | shape=(960, 960), nnz=30282
Loaded: M | shape=(960, 960), nnz=10098
Loaded: K | shape=(960, 960), nnz=30282
Loaded: M | shape=(960, 960), nnz=10098


In [5]:
# Build graphs from matrices
diag_K = A_K.diagonal()
diag_M = A_M.diagonal()

G_K = matrix_to_graph(A_K, symmetrize='sum', drop_diagonal=True, abs_weights=True, node_vweight='diag', diag=diag_K)
G_M = matrix_to_graph(A_M, symmetrize='sum', drop_diagonal=True, abs_weights=True, node_vweight='diag', diag=diag_M)

print(f"K: |V|={G_K.number_of_nodes()}, |E|={G_K.number_of_edges()}")
print(f"M: |V|={G_M.number_of_nodes()}, |E|={G_M.number_of_edges()} (vweights set from diagonal)")

K: |V|=960, |E|=14661
M: |V|=960, |E|=4569 (vweights set from diagonal)


In [6]:
# Partitioning configuration
# Modes: 'bipartition' | 'kway_recursive' | 'kway_metis'
PARTITION_MODE = 'kway_recursive'
NPARTS = 5  # keep 2 for classic bipartition

In [7]:
# Run partitions based on PARTITION_MODE and NPARTS
# Default params per graph
params_K = dict(weight='weight', initial_method='GGGP', refine_method='FM',
                balance_tol=0.03, max_levels=20, coarsen_limit=60, refine_passes=5, n_trials=6, seed=42)
params_M = dict(weight='weight', initial_method='COMPONENT_AWARE', refine_method='FM',
                balance_tol=0.02, max_levels=22, coarsen_limit=80, refine_passes=6, n_trials=8, seed=123)

if PARTITION_MODE == 'bipartition' and NPARTS == 2:
    part_K = multilevel_bipartition(G_K, **params_K)
    part_M = multilevel_bipartition(G_M, **params_M)
elif PARTITION_MODE == 'kway_recursive':
    part_K = k_way_partition(G_K, NPARTS, **params_K)
    part_M = k_way_partition(G_M, NPARTS, **params_M)
elif PARTITION_MODE == 'kway_metis':
    part_K = partition_graph_metis(G_K, nparts=NPARTS, weight='weight', seed=params_K.get('seed', 42))
    part_M = partition_graph_metis(G_M, nparts=NPARTS, weight='weight', seed=params_M.get('seed', 123))
else:
    raise ValueError(f"Unsupported configuration: PARTITION_MODE={PARTITION_MODE}, NPARTS={NPARTS}")

summarize_generic(G_K, part_K, f"K [{PARTITION_MODE} {NPARTS}]")
summarize_generic(G_M, part_M, f"M [{PARTITION_MODE} {NPARTS}]")

K [kway_recursive 5]: cut=8108.1271, parts=[0, 1, 2, 3, 4], per=[8759.235479002868, 8252.04372621818, 4180.907276973245, 7778.437583232111, 4675.240908415445], total=33645.8650
M [kway_recursive 5]: cut=11504.0178, parts=[0, 1, 2, 3, 4], per=[42047.22420873578, 24397.71386183873, 45520.04723864913, 45148.53699151335, 24109.36030632651], total=181222.8826


In [8]:
# Baseline METIS comparison for the same NPARTS
try:
    metis_part_K = partition_graph_metis(G_K, nparts=NPARTS, weight='weight', seed=42)
    summarize_generic(G_K, metis_part_K, f"METIS K [{NPARTS}]")
except Exception as e:
    print('METIS K failed:', e)

try:
    metis_part_M = partition_graph_metis(G_M, nparts=NPARTS, weight='weight', seed=123)
    summarize_generic(G_M, metis_part_M, f"METIS M [{NPARTS}]")
except Exception as e:
    print('METIS M failed:', e)

METIS K [5]: cut=8391.6201, parts=[0, 1, 2, 3, 4], per=[6706.458127358207, 6733.513206070527, 6729.453954886778, 6727.261356945957, 6749.178328580379], total=33645.8650
METIS M [5]: cut=13125.1420, parts=[0, 1, 2, 3, 4], per=[36247.69678502237, 35888.10202492381, 36593.69668642173, 36448.5468040664, 36044.84030662915], total=181222.8826
